# 03 - The bias sweep

**Purpose.** Turn session 01's frames into published constants: `pedestal(gain, offset)`,
`R(gain)` in ADC counts, the HCG threshold, and the offset this project fixes on. It also reads
the pedestal drift trace, which is what licenses - or forces - the bias interleaving in
session 02.

**What it is not for.** No conversion to electrons. That needs `g`, which the photon transfer
curve has not measured yet, and a number in electrons quoted before `g` exists is a number with
a made-up scale factor in it. Everything here is in ADC counts.

**Two halves, and they run at different times.** The first half *captures* - it talks to the
camera, and it is the only place in this project that does. The second half *reads what is on
disk* and publishes. They are separate cells on purpose: re-running the analysis must never
re-run the capture, because the frames cost a bench night and the camera does not always agree
to give them back.

The run sheet is `protocols/01-bias-sweep.md`. It is the authority on what is captured and why;
this notebook is the thing that does it, and it repeats the numbers rather than restating the
reasoning.

**Before anything runs:** read the `LEGACY.md` entries this session consumes - L01, L03, L04,
L05 in the pre-flight, then L10, L13, L25, L26, L27, L29 and L31 in the analysis. L01 in
particular is the difference between a night of measurement and a night of frames with a 17%
error baked into every one of them.


In [ ]:
import json, pathlib, sys, time
import datetime as dt

sys.path.insert(0, str(pathlib.Path.cwd().parent))

import numpy as np
import pandas as pd

from astropix import asi, fits as F, spatial as SP, stats as ST

pd.set_option("display.width", 200)

RESULTS = pathlib.Path("..") / "results"
DATA = pathlib.Path("..") / "data" / "session01"
FRAMES = DATA / "frames"
FRAMES.mkdir(parents=True, exist_ok=True)

SWEEP = RESULTS / "bias_sweep.csv"            # one row per (gain, offset)
CONSTANTS = RESULTS / "bias_constants.json"   # the scalars, with provenance
DRIFT = RESULTS / "pedestal_drift.csv"        # block 4, the 15-minute trace
INVENTORY = DATA / "inventory.csv"            # per frame; intermediate, not published
SPECS = pathlib.Path("..") / "vendor" / "asi585specs" / "gain-curves.csv"

# --- the capture plan, from protocols/01-bias-sweep.md ----------------------
ROI = (1408, 568, 1024, 1024)   # even throughout, width % 8 == 0 (L05)
DISCARD = 2                     # frames thrown away after every setting change
SETPOINT_C = asi.SETPOINT_C     # -10 C, and it is not a free parameter here

COARSE_GAINS = list(range(0, 601, 10))                              # 61
FINE_GAINS = [g for g in range(180, 221, 2) if g % 10]              # 16 new
OFFSET_ARM_OFFSETS = [0, 5, 10, 20, 25, 30, 35, 40, 45, 50]         # 15 is block 1
OFFSET_ARM_GAINS = [0, 50, 100, 190, 200, 252, 300, 600]
PROJECT_OFFSET = 15             # ZWO's recommendation: a hypothesis, not an inheritance

DRIFT_GAIN, DRIFT_N, DRIFT_PERIOD_S = 100, 450, 2.0   # 15 minutes at one frame per 2 s

planned = (len(COARSE_GAINS) * 20 + len(FINE_GAINS) * 20
           + len(OFFSET_ARM_OFFSETS) * len(OFFSET_ARM_GAINS) * 10 + DRIFT_N)
print(f"{planned} frames planned, {planned * ROI[2] * ROI[3] * 2 / 1e9:.1f} GB on C:")
print(f"{len(list(FRAMES.glob('*.fits')))} frames already on disk")


## Capture

### Opening the camera

Three things are read off the device rather than assumed, and all three go in the session
record: the control ranges, the minimum exposure, and the white balance the camera shipped
with. The retired project assumed gain ran 0-400 when it runs 0-600, so a fifth of the sweep it
believed it was taking did not exist (L05).

If this raises, the message names the two causes - a missing Windows *driver*, or the ASIAIR
powered on and holding the camera over USB - because they present identically as zero cameras
(L02).


In [ ]:
# A restart is the recovery plan, so the capture half starts from an empty
# directory or it does not start.  Half of a previous run mixed into this one is
# the kind of dataset that looks fine and is wrong.
existing = list(FRAMES.glob("*.fits"))
assert not existing, (f"{len(existing)} frames already in {FRAMES} -- delete the "
                      "directory to restart, or skip to the analysis half")

rig = asi.open_camera()

print("gain range     ", rig.range("Gain"))
print("offset range   ", rig.range("Offset"))
print("exposure range ", rig.range("Exposure"), "us")
print("white balance shipped:", rig.get("WB_R"), rig.get("WB_B"))

asi.neutralise_white_balance(rig)
asi.set_roi(rig, *ROI)
EXPOSURE_S = rig.min_exposure_s()      # measured, never assumed
print("\nWB now:", rig.get("WB_R"), rig.get("WB_B"))
print(f"bias exposure: {EXPOSURE_S * 1e6:.0f} us")


### Gate 1 - white balance, verified from the pixels (L01)

Setting `WB_R = WB_B = 50` and reading it back proves the *control* took. It does not prove the
pipeline honoured it, and that distinction cost the retired project every read-noise number it
ever measured.

The evidence is in the pixels. This camera digitises 12 bits into a 16-bit container, so the
spacing between adjacent reachable values is exactly 16. A white balance multiplies each plane
by its own factor and smears that grid: the greens stay at 16 while red reads 17 or 18 and blue
reads 24. So the modal step **must be 16 on all four planes**.

**Nothing captured before this passes is usable.** If it fails, stop the session. There is no
correcting it later - the information is gone.


In [ ]:
asi.configure(rig, gain=100, offset=PROJECT_OFFSET)
for _ in range(DISCARD):
    asi.capture(rig, EXPOSURE_S)

gate1 = {}
for _ in range(5):
    mosaic, _ = asi.capture(rig, EXPOSURE_S)
    for name, plane in SP.split(mosaic).items():          # stored values: the grid is 16 there
        gate1.setdefault(name, []).append(ST.value_step(plane))

for name in SP.PLANES:
    print(f"  {name:2s} modal step {gate1[name]}")

bad = {n: s for n, s in gate1.items() if set(s) != {16}}
assert not bad, (f"white balance is still being applied: {bad} -- stop the session, "
                 "nothing captured from here is usable (L01)")
print("\nGate 1 passed: the step is 16 on all four planes, on five frames.")


### Gate 2 - the cooler holds

-10 C, held in band for a continuous ten minutes before the first sweep frame. The window is
continuous and an excursion restarts it, because the first touch of a setpoint is the middle of
a swing rather than the end of one (L04).

Two details that are not fussiness. The cooler is judged by the *temperature trend*, not by the
duty cycle - a lookup on the wrong control name returned a hard 0% forever while the cooler was
working perfectly (L03), so duty cycle is a number that has already lied here once. And every
reading is written to disk as it is taken: the sensor reports a flat 0 with the cooler off, so an
interrupted cool-down takes its own answer with it.

Measured on 2026-08-28 on this rig: 14.0 C to -10.0 C in 589 s, mean 2.5 C/min, no ring, 62%
duty at setpoint. So expect roughly eight minutes of cooling and then the ten-minute window.

**The cooler does not survive `close()`, so the whole session is one kernel.** Measured the same
day: set `CoolerOn = 1`, close the camera, reopen, and it reads 0 - while `Gain` and `Offset`
come back exactly as they were left. Control values persist in the camera; the TEC is tied to
the open session. Cooling in one process and capturing in another is therefore impossible, and
if this kernel dies the cool-down starts over. It is also why "the camera is still cold" is
never a safe assumption between runs: with the cooler off the sensor reports a flat 0, so
nothing contradicts the belief.


In [ ]:
started = time.monotonic()
cool_log = open(DATA / "cooldown.csv", "w", newline="")
cool_log.write("elapsed_s,temp_C,duty_pct\n")


def show(elapsed, temp, duty):
    cool_log.write(f"{elapsed},{temp},{duty}\n")
    cool_log.flush()                     # the reading exists nowhere else
    if int(elapsed) % 30 == 0:
        print(f"  {elapsed:6.0f} s  {temp!s:>6} C  {duty:>3}%", flush=True)


try:
    trace = asi.cool_to(rig, SETPOINT_C, log=show)
finally:
    cool_log.close()

temps = [t for _, t, _ in trace if t is not None]
print(f"\nsettled in {trace[-1][0]:.0f} s;  {temps[0]} C -> {temps[-1]} C, "
      f"minimum {min(temps)} C")
print(f"duty at setpoint {trace[-1][2]}%  <- the headroom this room leaves")


### The capture loop

One function, and the notebook's only loop over frames - the library does one frame, this does
the sweep.

**A dead run is restarted, not resumed.** Delete `data/session01/frames/` and go again. The whole
capture is well under an hour, the cooler has to settle from scratch anyway, and resumption
logic is code that runs exactly once, in the dark, under time pressure, having never been tested
on the case it exists for. `F.write` refuses to overwrite, so a restart into a directory that
still has frames stops immediately rather than blending two runs.

**Temperature is recorded, not enforced.** Each frame's header carries the sensor reading taken
after its own exposure, so a frame that drifted out of band can be excluded in the analysis by
the evidence rather than by a rule applied blind at 2 a.m. What the loop does is *say so
immediately*, because a cooler failing at gain 200 is worth knowing before gain 600.


In [ ]:
def capture_run(gain, offset, n, tag):
    """`n` frames at one setting.  Returns how many came back out of band."""
    asi.configure(rig, gain=gain, offset=offset)
    for _ in range(DISCARD):
        asi.capture(rig, EXPOSURE_S)

    off_band = 0
    for i in range(n):
        mosaic, header = asi.capture(rig, EXPOSURE_S, imagetyp="BIAS")
        F.write(FRAMES / f"{tag}_g{gain:03d}_o{offset:03d}_{i:03d}.fits", mosaic, header)
        temp = header["CCD-TEMP"]
        if temp is None or abs(temp - SETPOINT_C) > asi.BAND_C:
            off_band += 1
            print(f"    ! gain {gain} frame {i} at {temp} C, out of band", flush=True)
    return off_band


def run_block(name, settings, n):
    """`settings` is a list of (gain, offset).  Progress is per setting: a block
    that reports only at the end is a block nobody can judge while it runs."""
    t0, flagged = time.monotonic(), 0
    for k, (gain, offset) in enumerate(settings, 1):
        flagged += capture_run(gain, offset, n, name)
        elapsed = time.monotonic() - t0
        print(f"  {name} {k:>3}/{len(settings)}  gain {gain:>3} offset {offset:>3}  "
              f"{elapsed / 60:5.1f} min elapsed, "
              f"{elapsed / k * (len(settings) - k) / 60:5.1f} to go", flush=True)
    print(f"{name}: {len(settings) * n} frames, {flagged} out of band, "
          f"{(time.monotonic() - t0) / 60:.1f} min\n", flush=True)


### Blocks 1-3

| block | offset | gains | frames each |
|---|---|---|---|
| 1 - coarse | 15 | 0 to 600 step 10 | 20 |
| 2 - fine | 15 | 180 to 220 step 2, the ten-multiples already done | 20 |
| 3 - offset arm | 0 to 50 step 5, without 15 | 0, 50, 100, 190, 200, 252, 300, 600 | 10 |

Block 2 samples the read-noise cliff densely, because ZWO put the high-conversion-gain
transition at 252 and the retired project measured it at 200 (L26). Both are predictions here,
and the fine grid exists so the data can disagree with both.

Block 3 walks the offset **down to 0 on purpose**. Offset 0 at gain 600 is expected to clip, and
that expectation is the measurement: the floor has to be crossed under observation rather than
guessed at. What gets reported is the clipped *fraction* against a 0.1% threshold - L13's
cautionary tale is a run where 58 pixels in a million read zero and were called a clipped
offset, when they were cold pixels.


In [ ]:
run_block("coarse", [(g, PROJECT_OFFSET) for g in COARSE_GAINS], 20)


In [ ]:
run_block("fine", [(g, PROJECT_OFFSET) for g in FINE_GAINS], 20)


In [ ]:
run_block("offsetarm", [(g, o) for o in OFFSET_ARM_OFFSETS for g in OFFSET_ARM_GAINS], 10)


### Block 4 - the pedestal drift trace

A different loop, because this one is paced: one frame every 2 s for 15 minutes, at gain 100 and
the project offset. The sweep above asks *what is the pedestal*; this asks *does it stay there*,
and the answer decides whether session 02 has to interleave bias frames between its darks or
merely may.

Cadence matters more than count. The frames are timestamped from `DATE-OBS`, so a late frame is
not a lost one, but a trace whose spacing wanders is a trace whose slope is harder to trust.


In [ ]:
stem = f"drift_g{DRIFT_GAIN:03d}_o{PROJECT_OFFSET:03d}"
asi.configure(rig, gain=DRIFT_GAIN, offset=PROJECT_OFFSET)
for _ in range(DISCARD):
    asi.capture(rig, EXPOSURE_S)

t0 = time.monotonic()
for i in range(DRIFT_N):
    mosaic, header = asi.capture(rig, EXPOSURE_S, imagetyp="BIAS")
    F.write(FRAMES / f"{stem}_{i:04d}.fits", mosaic, header)
    if i % 50 == 0:
        print(f"  {i:>3}/{DRIFT_N}  {header['CCD-TEMP']} C  "
              f"{(time.monotonic() - t0) / 60:.1f} min", flush=True)
    slack = (i + 1) * DRIFT_PERIOD_S - (time.monotonic() - t0)
    if slack > 0:
        time.sleep(slack)

print(f"drift trace: {DRIFT_N} frames in {(time.monotonic() - t0) / 60:.1f} min")


### Closing down

The cooler is switched off deliberately and the camera closed. A hard-killed process does switch
the TEC off, but leaving it that way by accident is the difference between "warmed up on
purpose" and a camera that behaves oddly next session for reasons nobody wrote down.

Let it warm before unplugging: condensation on a cold sensor is a hardware problem, not a data
one.


In [ ]:
rig.set("CoolerOn", 0, verify=False)
rig.close()
print("cooler off, camera closed.  Let it reach ambient before unplugging.")
print(f"{len(list(FRAMES.glob('*.fits')))} frames on disk in {FRAMES}")


## Analysis

Everything below reads from disk. The camera is not needed and must not be opened - this half is
re-run whenever the analysis changes, and the frames do not change with it.

Three rules fixed before the data existed, and repeated here so a reader does not have to take
them on trust:

1. Statistics on the **CFA mosaic split RGGB**, never debayered. Interpolated pixels have
   correlated noise and would silently invalidate every variance below.
2. Values in **ADC counts** = stored / 16.
3. `R` comes from the **difference of frame pairs**, not from a photon-transfer-curve intercept
   (L10). A pair difference cancels the fixed pattern exactly; an intercept extrapolates it in.


### One pass over the frames

Two products from one read, because reading 2,800 frames twice is a waste of the only expensive
thing here:

- **per-frame plane means** - the pedestal, and the clipping fraction at the floor;
- **per-pair difference sigmas** - `R`, from consecutive frames at the same setting.

The pair difference is taken as frames arrive, keeping only the previous frame's planes, so the
loop never holds more than two frames. `sigma / sqrt(2)` because differencing two independent
frames of equal noise doubles the variance.

Both a standard deviation and a MAD-based sigma are kept. They agree when the difference is
Gaussian and diverge when a handful of hot or telegraph pixels dominate, and which of those is
happening is worth knowing rather than deciding in advance.


In [ ]:
def measure(path):
    """Per-plane numbers for one frame, in ADC counts.  Reads the whole frame:
    the ROI is 1 Mpx, and a sweep constant should not be built on a sample."""
    mosaic, header = F.read(path)
    planes = {n: ST.to_adc(p).astype(np.float32) for n, p in SP.split(mosaic).items()}
    row = {
        "path": path.name,
        "tag": path.name.split("_")[0],
        "gain": header["GAIN"], "offset": header["OFFSET"],
        "exptime": header["EXPTIME"], "ccd_temp": header.get("CCD-TEMP"),
        "date_obs": header["DATE-OBS"],
    }
    for n, p in planes.items():
        row["mean_" + n.lower()] = float(p.mean())
        row["zero_" + n.lower()] = float(np.mean(p <= 0))
        row["sat_" + n.lower()] = float(np.mean(p >= ST.ADC_FULL_SCALE))
    return row, planes


paths = sorted(FRAMES.glob("*.fits"))
rows, pairs, previous = [], [], {}
t0 = time.monotonic()

for k, path in enumerate(paths, 1):
    row, planes = measure(path)
    rows.append(row)

    key = (row["tag"], row["gain"], row["offset"])
    if key in previous:                      # consecutive frames, same setting
        pair = {"tag": row["tag"], "gain": row["gain"], "offset": row["offset"]}
        for n, p in planes.items():
            d = p - previous[key][n]
            pair["sd_" + n.lower()] = float(d.std() / np.sqrt(2))
            pair["mad_" + n.lower()] = float(
                ST.MAD_TO_SIGMA * np.median(np.abs(d - np.median(d))) / np.sqrt(2))
        pairs.append(pair)
        del previous[key]                    # pairs are disjoint, not overlapping
    else:
        previous[key] = planes

    if k % 200 == 0:
        rate = k / (time.monotonic() - t0)
        print(f"  {k}/{len(paths)}  {rate:.0f} frames/s, "
              f"{(len(paths) - k) / rate / 60:.1f} min to go", flush=True)

frames = pd.DataFrame(rows)
pairs = pd.DataFrame(pairs)
frames.to_csv(INVENTORY, index=False)        # intermediate: data/, not results/
print(f"\n{len(frames)} frames, {len(pairs)} pairs, "
      f"{(time.monotonic() - t0) / 60:.1f} min")
print(frames.groupby('tag').size().to_string())


### Did the session hold together?

Before any constant, three questions whose wrong answers would poison everything after:
did every setting get the frames it was supposed to, did the temperature hold, and is the
container what we think it is.

A frame count short of plan is not automatically a problem - it is a problem if nobody noticed.


In [ ]:
PLANE_COLS = ["mean_" + p.lower() for p in SP.PLANES]

counts = frames.groupby(["tag", "gain", "offset"]).size()
print("frames per setting:")
print(counts.groupby("tag").agg(["min", "max", "size"]).to_string())

temp = pd.to_numeric(frames.ccd_temp, errors="coerce")
off_band = frames[(temp - SETPOINT_C).abs() > asi.BAND_C]
print(f"\ntemperature {temp.min()} .. {temp.max()} C, "
      f"{len(off_band)} frames outside +/-{asi.BAND_C} C of {SETPOINT_C}")

print(f"\nexposure: {sorted(frames.exptime.unique())} s")
print(f"planes agree to within "
      f"{(frames[PLANE_COLS].max(axis=1) - frames[PLANE_COLS].min(axis=1)).max():.2f} counts "
      "at the widest -- a zero-light frame whose planes disagree has a light leak")


### `pedestal(gain, offset)`

The plane mean of a zero-light frame, per setting. Then the fit L27 asks for:
`pedestal = A + B * amplification`, where `A` is the purely digital part - the offset control,
worth roughly 4 ADC counts per unit - and `B * amplification` is the analogue part that scales
with the gain the sensor applies. ZWO gain is in units of 0.1 dB, so amplification is
`10 ** (gain / 200)`.

This cell measures the table and the digital term. The analogue term needs the branch split, and
the branch split needs the HCG threshold, so the fit itself is two cells further down - **per
conversion-gain branch, never across the transition**, because a single fit spanning both lands
between them and mispredicts each by 8-17%.


In [ ]:
pedestal = (frames.groupby(["gain", "offset"])[PLANE_COLS].mean()
            .mean(axis=1).rename("pedestal").reset_index())

at_project = pedestal[pedestal.offset == PROJECT_OFFSET].sort_values("gain")
print(f"pedestal at offset {PROJECT_OFFSET}, ADC counts:")
print(at_project.set_index("gain")["pedestal"].to_string())

# the digital part: pedestal vs offset at fixed gain
arm = pedestal[pedestal.gain.isin(OFFSET_ARM_GAINS)]
print("\npedestal vs offset (rows: gain, cols: offset):")
print(arm.pivot(index="gain", columns="offset", values="pedestal").round(2).to_string())

per_offset_slope = {
    int(g): float(np.polyfit(d.offset, d.pedestal, 1)[0])
    for g, d in arm.groupby("gain") if len(d) > 2
}
print("\ncounts of pedestal per unit of offset, per gain:")
print(pd.Series(per_offset_slope).round(3).to_string())
print("L27 predicts ~4.0 and predicts it is gain-independent; both are testable here.")


### `R(gain)` - read noise in ADC counts

From pair differences, per plane, per gain, at the project offset. This is the number the whole
`R^2 / t` term of the SNR model rests on, and it is the reason the sub-exposure question has an
answer at all: `R` is the only noise source that does not grow with exposure time, so it is the
only one a longer sub-exposure dilutes.

**Only the sweep blocks feed it.** The drift block is 450 frames at gain 100 and the project
offset, so it matches the sweep's own setting exactly and would silently supply 225 of the 228
pairs at that one gain - one point of the curve measured a hundred times more densely than its
neighbours, and measured over fifteen minutes rather than one. It is excluded here and used
below as what it actually is: an independent check on gain 100, from a different block.

`R_err` is the scatter **across pairs**, divided by the root of their number: the uncertainty a
published constant carries. `plane_spread` is a different question - whether the four CFA planes
read alike - and answering one with the other would understate the first and hide the second.


In [ ]:
SD_COLS = ["sd_" + p.lower() for p in SP.PLANES]
MAD_COLS = ["mad_" + p.lower() for p in SP.PLANES]
SWEEP_TAGS = ("coarse", "fine")

sweep_pairs = pairs[(pairs.offset == PROJECT_OFFSET) & pairs.tag.isin(SWEEP_TAGS)].copy()
sweep_pairs["R"] = sweep_pairs[SD_COLS].mean(axis=1)      # one number per pair
by_gain = sweep_pairs.groupby("gain")

read_noise = pd.DataFrame({
    "n_pairs": by_gain.size(),
    "R_sd": by_gain.R.mean(),
    "R_mad": sweep_pairs.assign(m=sweep_pairs[MAD_COLS].mean(axis=1)).groupby("gain").m.mean(),
    "R_err": by_gain.R.std() / np.sqrt(by_gain.size()),
    "plane_spread": (by_gain[SD_COLS].mean().max(axis=1)
                     - by_gain[SD_COLS].mean().min(axis=1)),
})
print(read_noise.round(3).to_string())
print("\nsd vs mad: they agree on Gaussian noise and diverge when a few hot pixels dominate.")

check = pairs[pairs.tag == "drift"]
if len(check):
    R_drift = float(check[SD_COLS].mean(axis=1).mean())
    R_sweep = float(read_noise.R_sd.get(DRIFT_GAIN, np.nan))
    print(f"\ngain {DRIFT_GAIN}: sweep {R_sweep:.4f}, drift block {R_drift:.4f} "
          f"({len(check)} pairs) -- {abs(R_drift - R_sweep) / R_sweep:.2%} apart")


### The HCG threshold

The high-conversion-gain transition is a **step down** in read noise between two adjacent gains:
the sensor switches to a larger conversion gain and the same electron produces more counts, so
the read noise expressed in electrons falls even though the counts do not.

**ZWO's own published chart annotates `HCG = 200`** (`vendor/asi585specs/`), which is also what
the retired project measured (L26). The project had been carrying "ZWO say 252" as a competing
prediction; it has no source, and 252 is the ASI2600's threshold. So there is one prediction
here, not two, and the fine grid straddles it.

The grid samples every 2 gain units from 180 to 220. If the cliff is not in that window the
honest answer is to widen the grid on another night, not to promote the largest wobble to a
threshold.


In [ ]:
r = read_noise.R_sd.sort_index()
step = (r.diff() / r.shift()).rename("fractional change")   # negative = a drop
fine = step[(step.index >= 178) & (step.index <= 222)]
print("fractional change in R between adjacent gains, around the fine grid:")
print(fine.round(4).to_string())

hcg_gain = int(step.idxmin())
hcg_drop = float(step.min())
print(f"\nlargest drop: {hcg_drop:+.1%} entering gain {hcg_gain}")
print("  predicted: 200, by ZWO's chart and by the retired project alike (L26)")

if not (178 <= hcg_gain <= 222) or hcg_drop > -0.10:
    print("\n  No convincing cliff in the fine window.  That is a result: widen the grid\n"
          "  on another night rather than promoting the largest wobble to a threshold.")


### Against ZWO's published curves

`vendor/asi585specs/gain-curves.csv` is ZWO's chart read into a table, and it is a **hypothesis**
- read off a plot by eye, ±5% at best, and no substitute for the measurement above.

The comparison is only possible in one direction. ZWO quote read noise in electrons; this
session works in ADC counts, because electrons need `g` and `g` is the photon transfer curve,
which has not run. So the prediction compared here is `R_adu = R_e / g`, using ZWO's *own* `g`
curve - which makes it a **joint** test of two published curves at once. A disagreement cannot
be blamed on either one until session 03 measures `g` independently.

Note also that ZWO's chart stops at gain 450 while this camera's control runs to 600. A fifth of
the coarse sweep is measuring territory no vendor curve covers, and there is nothing to compare
it against.


In [ ]:
spec = pd.read_csv(SPECS)
# ZWO tabulate both branches at the transition; keep the one this session measured
spec = spec[spec.branch.values == np.where(spec.gain >= hcg_gain, "hcg", "lcg")]

cmp = spec.set_index("gain")[["read_noise_adu_predicted"]].join(read_noise[["R_sd", "R_err"]])
cmp["ratio"] = cmp.R_sd / cmp.read_noise_adu_predicted
print("read noise in ADC counts, measured against ZWO's R_e / g:")
print(cmp.dropna().round(3).to_string())
print("\nAgreement to a few per cent would mean both published curves are right *together*.\n"
      "A consistent ratio away from 1 is the more interesting outcome: it points at `g`,\n"
      "and session 03 is what separates the two.")


### The pedestal fit, per branch

Now the threshold exists, the analogue term can be fitted: `pedestal = A + B * amplification`,
with `amplification = 10 ** (gain / 200)` because ZWO gain is in units of 0.1 dB, so gain 200 is
20 dB and a factor of ten.

The fit is done **twice, once per conversion-gain branch**, and then once across both purely to
show what that costs. L27's claim is that the single fit lands between the branches and
mispredicts each by 8-17%; the residual columns below are that claim being checked rather than
repeated. A pedestal is subtracted from every frame this project ever calibrates, so a fit that
is wrong by 8% is wrong by 8% in every dark, every flat and every light.


In [ ]:
def fit(d):
    """`pedestal = A + B * amplification`, returned with its worst residual."""
    amp = 10 ** (d.gain / 200.0)
    B, A = np.polyfit(amp, d.pedestal, 1)
    resid = d.pedestal - (A + B * amp)
    return {"A": float(A), "B": float(B),
            "max_resid": float(resid.abs().max()),
            "max_resid_pct": float((resid / d.pedestal).abs().max()),
            "n": int(len(d))}


branch = at_project.assign(hcg=at_project.gain >= hcg_gain)
fits = {("hcg" if h else "lcg"): fit(d) for h, d in branch.groupby("hcg") if len(d) > 2}
fits["single fit, both branches"] = fit(branch)

print(f"amplification = 10 ** (gain / 200);  branch split at gain {hcg_gain}\n")
print(pd.DataFrame(fits).T.to_string(float_format=lambda v: f"{v:.4f}"))
print("\nL27 predicts the single fit mispredicts each branch by 8-17%.  The "
      "`max_resid_pct`\nrow above is that prediction meeting the data.")


### Clipping, and the offset the project fixes on

Clipping is reported as a **fraction**, against 0.1%, and beside it the distance from the
pedestal down to zero measured in read noises. L13's failure was a run where 58 zero-valued
pixels in a million were read as a clipped offset; they were cold pixels, and the distribution
was nowhere near the floor. A distance of many `R` with a handful of zeros is defective pixels.
A distance of one or two `R` is a genuinely clipped offset, and the frames are unusable.

The offset this project fixes on is the smallest one whose clipped fraction stays under 0.1% at
every gain it intends to use - with margin, and with a number behind the margin.


In [ ]:
ZERO_COLS = ["zero_" + p.lower() for p in SP.PLANES]
CLIP_THRESHOLD = 0.001          # 0.1% (L13)

clip = (frames.groupby(["offset", "gain"])[ZERO_COLS].mean().mean(axis=1)
        .rename("zero_frac").reset_index())
print("clipped fraction (rows: offset, cols: gain):")
print(clip.pivot(index="offset", columns="gain", values="zero_frac")
      .map(lambda v: f"{v:.2e}").to_string())

# how far the distribution sits above the floor, in read noises
ped = pedestal.set_index(["gain", "offset"]).pedestal
headroom = pd.DataFrame([
    {"offset": o, "gain": g,
     "pedestal": ped.get((g, o)),
     "R": read_noise.R_sd.get(g),
     "sigmas_above_zero": (ped.get((g, o)) / read_noise.R_sd.get(g)
                           if read_noise.R_sd.get(g) else np.nan)}
    for o, g in zip(clip.offset, clip.gain)])
print("\ndistance from zero, in read noises (rows: offset, cols: gain):")
print(headroom.pivot(index="offset", columns="gain", values="sigmas_above_zero")
      .round(1).to_string())

safe = [o for o, d in clip.groupby("offset") if (d.zero_frac < CLIP_THRESHOLD).all()]
chosen = min(safe) if safe else None
print(f"\noffsets clipping under {CLIP_THRESHOLD:.1%} at every gain measured: {safe}")
print(f"smallest safe offset: {chosen}   (ZWO recommends {PROJECT_OFFSET})")


### Pedestal drift over 15 minutes

The plane mean against time, with sensor temperature beside it. The question is not whether the
pedestal moves - everything moves - but whether it moves by more than the frame-to-frame scatter
over the interval that separates a dark from its bias in session 02.

If drift is below the scatter, interleaving is a precaution and its cost can be reduced. If it is
measurable, interleaving is mandatory there **and** in the session 03 bias pairs. This is
exactly the failure that produced a negative dark current once: a master bias taken four hours
from the darks it was subtracted from.


In [ ]:
d = frames[frames.tag == "drift"].copy()
d["t"] = pd.to_datetime(d.date_obs)
d = d.sort_values("t")
d["elapsed_s"] = (d.t - d.t.iloc[0]).dt.total_seconds()
d["pedestal"] = d[PLANE_COLS].mean(axis=1)

slope, intercept = np.polyfit(d.elapsed_s, d.pedestal, 1)
resid = d.pedestal - (slope * d.elapsed_s + intercept)
scatter = float(d.pedestal.diff().abs().median())      # frame to frame

print(f"{len(d)} frames over {d.elapsed_s.max() / 60:.1f} min at "
      f"{d.elapsed_s.diff().median():.2f} s cadence")
print(f"drift        {slope * 60:+.4f} counts/min  "
      f"({slope * 60 * 15:+.3f} over the trace)")
print(f"residual     {resid.std():.4f} counts about the line")
print(f"frame-to-frame scatter {scatter:.4f} counts")
print(f"temperature  {d.ccd_temp.min()} .. {d.ccd_temp.max()} C")
print("\ninterleaving in session 02 is "
      + ("MANDATORY: the drift exceeds the frame-to-frame scatter."
         if abs(slope * 60 * 15) > scatter else
         "a precaution: the drift over 15 min is below the frame-to-frame scatter."))

d[["elapsed_s", "date_obs", "pedestal", "ccd_temp"]].to_csv(DRIFT, index=False)
print(f"\nwrote {DRIFT}")


### Publishing

Two files. `bias_sweep.csv` is the table - one row per `(gain, offset)`, carrying the pedestal,
the read noise and the clipped fraction. `bias_constants.json` holds the scalars, and every one
of them carries `value, unit, uncertainty, source_frames, measured_on, notebook`, because the
model refuses to run on a constant that does not.

The provenance is not bookkeeping. A constant without its source frames cannot be re-measured
when it turns out to be wrong, and constants do turn out to be wrong.


In [ ]:
table = (pedestal.merge(read_noise.reset_index(), on="gain", how="left")
         .merge(clip, on=["gain", "offset"], how="left"))
table.to_csv(SWEEP, index=False)
print(f"wrote {SWEEP}: {len(table)} rows")
print(table.head(10).round(4).to_string())


In [ ]:
measured_on = dt.datetime.now(dt.timezone.utc).strftime("%Y-%m-%d")
n_frames = int(len(frames))


def constant(value, unit, uncertainty, note):
    return {"value": value, "unit": unit, "uncertainty": uncertainty,
            "source_frames": n_frames, "measured_on": measured_on,
            "notebook": "03_bias_sweep.ipynb", "note": note}


R_at_hcg = float(read_noise.R_sd.get(hcg_gain, np.nan))
constants = {
    "hcg_threshold_gain": constant(
        hcg_gain, "ZWO gain units (0.1 dB)", 2,
        "lowest gain of the high-conversion-gain branch; the fine grid steps by 2, "
        "which is the uncertainty.  ZWO predict 252, the retired project measured 200 (L26)"),
    "read_noise_at_hcg": constant(
        round(R_at_hcg, 4), "ADC counts", round(float(read_noise.R_err.get(hcg_gain, np.nan)), 4),
        "pair-difference sigma / sqrt(2), mean over four CFA planes (L10)"),
    "project_offset": constant(
        chosen, "offset units", 5,
        f"smallest offset clipping under {CLIP_THRESHOLD:.1%} at every gain measured; "
        f"the arm steps by 5.  ZWO recommend {PROJECT_OFFSET} (L13)"),
    "pedestal_per_offset_unit": constant(
        round(float(np.mean(list(per_offset_slope.values()))), 4), "ADC counts per offset unit",
        round(float(np.std(list(per_offset_slope.values()))), 4),
        "the digital term A; L27 predicts ~4.0 and gain-independent"),
    "pedestal_fit": constant(
        {k: {"A": round(v["A"], 4), "B": round(v["B"], 6)}
         for k, v in fits.items() if k in ("lcg", "hcg")},
        f"ADC counts, at offset {PROJECT_OFFSET}",
        {k: round(fits[k]["max_resid"], 4) for k in fits if k in ("lcg", "hcg")},
        "pedestal = A + B * 10**(gain/200), fitted per conversion-gain branch, "
        f"split at gain {hcg_gain}; uncertainty is the worst residual (L27)"),
    "pedestal_drift_rate": constant(
        round(float(slope * 60), 5), "ADC counts per minute", round(float(resid.std()), 5),
        f"gain {DRIFT_GAIN}, offset {PROJECT_OFFSET}, {len(d)} frames over 15 min at -10 C"),
    "bias_exposure": constant(
        float(frames.exptime.min()), "s", 0.0,
        "the camera's minimum exposure, read from the Exposure control"),
    "setpoint": constant(
        SETPOINT_C, "C", asi.BAND_C,
        "held for a continuous 10 minutes before the first frame; uncertainty is "
        "the sensor's own 0.5 C quantisation"),
}
with open(CONSTANTS, "w") as fh:
    json.dump(constants, fh, indent=2)
print(f"wrote {CONSTANTS}")
print(json.dumps(constants, indent=2))


### What this session decided

Fill this in from the numbers above, in the session record, before the frames are archived:

- Is the offset axis **retired**? It is, if `A` scales linearly with offset and `R` is
  offset-independent to better than 0.5%. Fix it once and never sweep it again.
- Where is the **HCG threshold**, and did either prediction survive?
- Does session 02 have to **interleave**?
- Which `LEGACY` entries did this check - L10, L13, L25, L26, L27, L29, L31 - and where does
  each land now that it has been checked? An entry that has been verified and left in the queue
  is a step that has not finished.

The one thing this notebook must not do is convert any of it to electrons. That is `g`, that is
the photon transfer curve, and that is a different night.
